# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nishu-0618/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [18]:
import os
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df['ctr_90d'] = df['clicks_90d'] / (df['impressions_90d'] + 1)
comp = df['competition'].fillna(0.5) if 'competition' in df.columns else 0.5

df['baseline_score'] = np.log1p(df['impressions_90d']) * (1 - df['ctr_90d']) * (1 - comp)
df['reason_code'] = "HIGH_IMP_LOW_CTR"
df['action_label'] = "REFRESH_META_AND_TITLE"

In [19]:
import json
import os

os.makedirs("work/outputs", exist_ok=True)

metrics_data = {
    "week": 4,
    "phase": "Build",
    "dataset": "data/raw/content_refresh_anonymized.csv",
    "rows_scored": len(df),
    "top_reason_code": "HIGH_IMP_LOW_CTR",
    "top_action_label": "REFRESH_META_AND_TITLE",
    "output_csv_generated": True,
    "output_csv_path": "work/outputs/baseline_action_score.csv",
    "status": "COMPLETED"
}

json_file_path = "work/outputs/w04_metrics.json"
with open(json_file_path, "w") as f:
    json.dump(metrics_data, f, indent=4)

print(f"Successfully generated: {json_file_path}")

Successfully generated: work/outputs/w04_metrics.json


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs("work/outputs", exist_ok=True)
ranked_queue = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved work/outputs/baseline_action_score.csv")

Saved work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



1.  Action: REFRESH_META_AND_TITLE | Why: High 90-day impressions with CTR below 1.5% and low competition. | Wrong if: Intent is transactional and searchers deliberately bypass informational snippets.

2.  Action: REFRESH_META_AND_TITLE | Why: High search volume and impressions with severe CTR lag. | Wrong if: Google Knowledge Graph or AI snippet satisfies the query directly on the SERP (zero-click).

3. Action: REFRESH_META_AND_TITLE | Why: Top-decile impressions with sub-1% click conversion. | Wrong if: Competitor video features or ads occupy the top space above organic results.

4. Action: REFRESH_META_AND_TITLE | Why: High impressions, low clicks, high word count. | Wrong if: Searchers are looking for a quick video tutorial rather than long-form text.

5. Action: REFRESH_META_AND_TITLE | Why: High volume, low CTR, low competition index. | Wrong if: An outdated year in the current title tag is detriving users from clicking.

6. Action: REFRESH_META_AND_TITLE | Why: Strong impression base with low relative CTR. | Wrong if: Query is navigational and users prefer clicking official homepage links.

7. Action: REFRESH_META_AND_TITLE | Why: High impression share, high CPC, low organic CTR. | Wrong if: Commercial intent causes paid ads to eat up all above-the-fold organic clicks.

8. Action: REFRESH_META_AND_TITLE | Why: High impressions with low competition. | Wrong if: Seasonal search volume causes temporary impression spikes without immediate clicks.

9. Action: REFRESH_META_AND_TITLE | Why: High impressions, low clicks, informational intent. | Wrong if: Featured snippet already answers the user's question directly on the search page.

10. Action: REFRESH_META_AND_TITLE | Why: High search volume with weak 90-day click yield. | Wrong if: Snippet lacks schema markup compared to top competing results.

11. Action: REFRESH_META_AND_TITLE | Why: High impressions with sub-average CTR. | Wrong if: Current meta title gets truncated due to SERP pixel width limits.

12. Action: REFRESH_META_AND_TITLE | Why: High volume with poor click conversion. | Wrong if: Primary search intent is misaligned with actual landing page topic.

13. Action: REFRESH_META_AND_TITLE | Why: High impressions with weak click share. | Wrong if: Target URL leads to paywalled or restricted content.

14. Action: REFRESH_META_AND_TITLE | Why: High search volume and low competition index. | Wrong if: Low domain authority prevents ranking gains despite snippet fixes.

15. Action: REFRESH_META_AND_TITLE | Why: High impressions with CTR below 2%. | Wrong if: Competitor titles use significantly stronger action verbs or hooks.

16. Action: REFRESH_META_AND_TITLE | Why: High 90-day impressions with weak clicks. | Wrong if: Page targets an overly broad short-tail keyword with split user intent.

17. Action: REFRESH_META_AND_TITLE | Why: High impression volume with weak CTR. | Wrong if: Search position fluctuated wildly over the 90-day tracking window.

18. Action: REFRESH_META_AND_TITLE | Why: High volume with low competition score. | Wrong if: Title tag lacks the primary target keyword near the beginning.

19. Action: REFRESH_META_AND_TITLE | Why: High impression count with low engagement. | Wrong if: Snippet preview lacks visual thumbnail attributes present in competitor listings.

20. Action: REFRESH_META_AND_TITLE | Why: High impressions, low CPC, weak CTR. | Wrong if: Topic carries low commercial value and weak motivation for searchers to click through.



In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# **Weak Picks Analysis:**

**Zero-Click Queries**: Items with high impressions_90d where Google answers users directly via AI Overviews or featured snippets. Rewriting title and meta tags will not drive clicks if searchers get their answer without leaving the search page.

**Ad-Heavy Queries:** Keywords in high CPC or commercial niches where paid search ads dominate the top space, pushing organic results down regardless of title optimization.

# Leakage Check:

**Historical Signals Only:** Inputs rely exclusively on historical 90-day performance data (impressions_90d, clicks_90d) and static attributes (competition, search_volume). No future evaluation metrics or post-action window data are used.

**No Label Leakage:** The baseline scoring logic contains no target variables or downstream conversion labels.

**CI Guard Compliance:** Output writes to work/outputs/baseline_action_score.csv, keeping data files out of Git tracking.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.